# `AAWindowSampler.sample_synthetic()`

Generate synthetic control windows from a configurable amino-acid prior. Useful as a third-distribution control alongside `sample_same_protein` (negatives) and `sample_different_protein` (unlabeled). Synthetic windows have no source protein, so `output_mode` is fixed to `'segments'`.

In [1]:
import aaanalysis as aa
import pandas as pd
aa.options["verbose"] = False

df_seq = pd.DataFrame({
    "entry":    ["P1", "P2"],
    "sequence": ["ACDEFGHIKLMNPQRSTVWY" * 2,
                 "YWVTSRQPNMLKIHGFEDCA" * 2],
    "pos":      [[10], [15]],
})
sampler = aa.AAWindowSampler(random_state=0)

First call — anchor schema. The eight-column `segments` schema is shared across all `AAWindowSampler` methods; synthetic rows use `entry_win = synth_{i}` and leave `entry` empty since there is no source protein.

In [2]:
df = sampler.sample_synthetic(df_seq=df_seq, n=5, window_size=9, seed=0)
aa.display_df(df=df, show_shape=True)

DataFrame shape: (5, 8)


,entry_win,entry,sequence,window,source_position,label,role,strategy
1,synth_0,,,PGAATWPRM,-1,0,Control,synthetic:global_freq
2,synth_1,,,WTAVAREVM,-1,0,Control,synthetic:global_freq
3,synth_2,,,GKADQPPIY,-1,0,Control,synthetic:global_freq
4,synth_3,,,YQQQIDRMH,-1,0,Control,synthetic:global_freq
5,synth_4,,,LVWINHNHI,-1,0,Control,synthetic:global_freq


## `df_seq`, `pos_col`

`df_seq` is consumed differently per generator — see the `generator` section below. `pos_col` is required only for the `'position_specific'` and `'scrambled'` generators (which read test windows from the positions in `pos_col`). For all other generators, `pos_col` is optional and is used only as the source of test windows for the anti-leakage filter.

## `n`, `window_size`

Total number of synthetic windows (`n`) and residue length per window (`window_size`):

In [3]:
df = sampler.sample_synthetic(df_seq=df_seq, n=3, window_size=12,
                              generator="global_freq", seed=0)
aa.display_df(df=df[["entry_win", "window"]], show_shape=True)

DataFrame shape: (3, 2)


,entry_win,window
1,synth_0,PGAATWPRMWTA
2,synth_1,VAREVMGKADQP
3,synth_2,PIYYQQQIDRMH


## `generator`

The synthesis recipe. Accepts three shapes:

- `str` — a built-in generator (`'uniform'`, `'global_freq'`, `'position_specific'`, `'scrambled'`) or an AAontology preset name (e.g. `'aa_composition'`, `'alpha_helix'`).
- `list[str]` — at least two distinct preset names; their per-AA priors are combined into a multiplicative joint prior.
- `dict[str, float]` — a custom single-character → probability table. Keys define the alphabet, so the sampler is not restricted to amino acids.

**Built-in generators** draw uniformly over the canonical 20 (`'uniform'`), from the empirical AA frequency in `df_seq` (`'global_freq'`), per-position in the test windows (`'position_specific'`), or by shuffling a test window (`'scrambled'`):

In [4]:
df = sampler.sample_synthetic(df_seq=df_seq, n=3, window_size=9,
                              generator="global_freq", seed=0)
aa.display_df(df=df[["window", "strategy"]], show_shape=True)

DataFrame shape: (3, 2)


,window,strategy
1,PGAATWPRM,synthetic:global_freq
2,WTAVAREVM,synthetic:global_freq
3,GKADQPPIY,synthetic:global_freq


**AAontology preset generators** use a curated scale as the per-AA prior. Composition presets (`'aa_composition'`, `'aa_composition_surface'`, `'aa_composition_mp'`) are true AA-frequency distributions; conformation presets (`'alpha_helix'`, `'beta_sheet'`, `'beta_strand'`, `'beta_turn'`, `'coil'`, `'linker'`, `'pi_helix'`) are normalized propensities used as physicochemically-biased priors:

In [5]:
df = sampler.sample_synthetic(df_seq=df_seq, n=3, window_size=9,
                              generator="alpha_helix", seed=0)
aa.display_df(df=df[["window", "strategy"]], show_shape=True)

DataFrame shape: (3, 2)


,window,strategy
1,NFAASWMQM,synthetic:alpha_helix
2,WSATAREVM,synthetic:alpha_helix
3,GKADPPNIY,synthetic:alpha_helix


**Mixed-prior generator** — a list of at least two distinct preset names combines their per-AA priors via element-wise product followed by renormalization. Useful e.g. for a membrane-helix prior (`['aa_composition_mp', 'alpha_helix']`):

In [6]:
df = sampler.sample_synthetic(df_seq=df_seq, n=3, window_size=9,
                              generator=["aa_composition_mp", "alpha_helix"], seed=0)
aa.display_df(df=df[["window", "strategy"]], show_shape=True)

DataFrame shape: (3, 2)


,window,strategy
1,MFAASVLQL,synthetic:mix:a..._mp+alpha_helix
2,VSATAQETL,synthetic:mix:a..._mp+alpha_helix
3,GIACNMMIY,synthetic:mix:a..._mp+alpha_helix


**Custom-alphabet generator** — a `dict[str, float]` over any single-character alphabet (e.g. DNA). The only generator path that produces non-amino-acid windows. Keys are case-sensitive and values must sum to 1.0 (within 1e-6):

In [7]:
df = sampler.sample_synthetic(df_seq=df_seq, n=3, window_size=8,
                              generator={"A": 0.25, "C": 0.25, "G": 0.25, "T": 0.25},
                              seed=0)
aa.display_df(df=df[["window", "strategy"]], show_shape=True)

DataFrame shape: (3, 2)


,window,strategy
1,GCAATTGG,synthetic:custom:A+C+G+T
2,GTTATAGA,synthetic:custom:A+C+G+T
3,TGCCAAGG,synthetic:custom:A+C+G+T


## `label_ref`, `role`

Semantic tags applied to all synthetic rows. Defaults: `role='Control'`, `label_ref=0`. Override for non-PU-learning workflows:

In [8]:
df = sampler.sample_synthetic(df_seq=df_seq, n=3, window_size=9,
                              generator="global_freq",
                              role="background", label_ref=-1, seed=0)
aa.display_df(df=df[["window", "role", "label"]], show_shape=True)

DataFrame shape: (3, 3)


,window,role,label
1,PGAATWPRM,background,-1
2,WTAVAREVM,background,-1
3,GKADQPPIY,background,-1


## `seed`

Per-call seed; falls back to the class-level `random_state` set at construction. A fixed seed yields deterministic output:

In [9]:
df_a = sampler.sample_synthetic(df_seq=df_seq, n=3, window_size=9,
                                generator="global_freq", seed=42)
df_b = sampler.sample_synthetic(df_seq=df_seq, n=3, window_size=9,
                                generator="global_freq", seed=42)
print("deterministic:", list(df_a["window"]) == list(df_b["window"]))

deterministic: True


Synthetic outputs use `entry_win = synth_{i}` with a per-call counter — concatenating multiple `sample_synthetic` outputs may collide on `entry_win`. Deduplicate on the `window` column instead.